# 🧠 burst_scan — does the classifier add direction? (gates a–f)

A detector that finds *movement* is not a strategy: `burst_scan_events.ipynb` §5 shows that bigger bursts mean
bigger moves in **both** directions. This notebook asks the second question, per sub-strategy: **can a model,
trained only on the outcomes of past events, pick the events whose move goes that side's way?** Everything is
out of sample: month *m* is scored by a model fit only on events whose outcome finished before *m* began.

| § | question |
|---|---|
| 1 | the folds — how much each month's model saw, and a **purge audit** that flips every label still running at a fold's start |
| 2 | **discrimination** — out-of-sample AUC per month and pooled, logistic regression vs gradient boosting |
| 3 | **calibration** — when the model says 30%, do 30% of those events win? |
| 4 | **selection** — realized win rate and demeaned return by decile of the prediction, gross and after costs |
| 5 | **what it learned** — coefficients per fold (do their signs hold?), importances, long vs short |
| 6 | **controls** — take-every-event, a model trained on placebo bars, shuffled outcomes (the leak test), and a label-as-feature canary |
| 7 | **stability** — by month, without the top five names, and the overnight share |
| 8 | **gates a–f and the verdict**, per side |

The gates (pre-registered in the README) read the classifier's **picks**: every out-of-sample event whose
probability clears its fold's threshold. The book's own P&L — spacing, clashes and the engine's rules applied —
is the optimize and diagnostics notebooks' job.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from trading_simple import backtest as bt, plots, strategies
from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
# research.py holds every computation the five notebooks share; signals() never imports it.
_spec = importlib.util.spec_from_file_location(
    "burst_research", strategies.STRATEGIES_DIR / "burst_scan" / "research.py"
)
rs = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(rs)
strat, S = rs.STRAT, rs.S  # the loaded strategy and its module's functions
compose = S["compose"]
SIDES = ("long", "short")
COLOR = {"long": plots._LONG, "short": plots._SHORT}


def below(fig, y=-0.1):
    """Legend under the plot: the default top-right legend sits on the subplot titles."""
    fig.update_layout(
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": y, "yanchor": "top"}
    )
    return fig


SUBSET = None  # e.g. 40 for a quick first run; None = all 200 names
OVERRIDES = {}  # any signals() knob
PRM = {**strat.params, **OVERRIDES}
SEED = 0
MODELS = ("lr", "gbm")

## 1 · The folds and the purge audit

`compose()` runs the strategy's own walk-forward for both models. The purge audit re-fits three folds after
**flipping every label that was still running at the fold's start**: a correctly purged walk-forward never saw
those labels, so its predictions for the fold must come out *identical*.

In [ ]:
panel = rs.load(settings, engine, subset=SUBSET)
runs = {m: compose(panel, **{**PRM, "model": m}) for m in MODELS}
ctx = runs["lr"]["ctx"]
fees, slip = bt.strategy_costs(strat, panel, settings)
RT = 2 * (fees + slip)  # one round trip, as a return
print(
    f"panel: {panel.close.shape[0]:,} {panel.bar} bars x {len(panel.symbols)} names · "
    f"round trip {1e4 * RT:.1f} bp at ${strat.order_notional:,.0f} orders"
)
T = {m: {s: runs[m]["events"][s] for s in SIDES} for m in MODELS}
month, first = S["month_starts"](ctx["index"], ctx["tz"])


def label(fm):
    return str(pd.Period(ctx["index"][first[fm]].tz_convert(ctx["tz"]).tz_localize(None), "M"))


folds = []
for side in SIDES:
    t = T["lr"][side]
    for fm, g in t[t["fold"] >= 0].groupby("fold"):
        folds.append(
            {
                "side": side,
                "month": label(fm),
                "train events": int(g["n_train"].iloc[0]),
                "test events": len(g),
                "picked": int(g["picked"].sum()),
                "win rate (test)": g["win"].mean(),
                "threshold": g["thr"].iloc[0],
            }
        )
display(pd.DataFrame(folds).round(3))
audit = pd.concat([rs.purge_audit(T["lr"][s], ctx, PRM, folds=3).assign(side=s) for s in SIDES])
audit

## 2 · Discrimination: out-of-sample AUC (Fig 2)

AUC is the chance that a random winning event is ranked above a random losing one: 0.5 is a coin, 1.0 is perfect.
Per month, pooled, and **within-month** (the mean of the monthly AUCs). The pooled number also rewards predicting
*which months* have more winners — a base rate that drifts — so the within-month figure is the one that measures
picking events. A model with skill sits above 0.5 in most months.

In [ ]:
aucs = []
for m in MODELS:
    for side in SIDES:
        t = T[m][side]
        o = t[t["p"].notna()]
        aucs.append(
            {
                "model": m,
                "side": side,
                "month": "pooled",
                "auc": rs.auc(o["p"], o["win"]),
                "within-month auc": rs.monthly_auc(o["p"], o["win"], o["fold"]),
                "events": len(o),
            }
        )
        for fm, g in o.groupby("fold"):
            aucs.append(
                {
                    "model": m,
                    "side": side,
                    "month": label(fm),
                    "auc": rs.auc(g["p"], g["win"]),
                    "events": len(g),
                }
            )
aucs = pd.DataFrame(aucs)
fig = go.Figure()
for m, dash in zip(MODELS, ("solid", "dash"), strict=True):
    for side in SIDES:
        g = aucs[(aucs["model"] == m) & (aucs["side"] == side) & (aucs["month"] != "pooled")]
        fig.add_trace(
            go.Scatter(
                x=g["month"],
                y=g["auc"],
                mode="lines+markers",
                name=f"{side} · {m}",
                line={"color": COLOR[side], "dash": dash},
            )
        )
fig.add_trace(
    go.Scatter(
        x=sorted(aucs.loc[aucs["month"] != "pooled", "month"].unique()),
        y=[0.5] * aucs.loc[aucs["month"] != "pooled", "month"].nunique(),
        mode="lines",
        name="coin (0.5)",
        line={"color": plots._MUTED, "width": 1},
    )
)
plots._base(
    fig, title="Fig 2 · out-of-sample AUC by month (solid = logistic, dash = boosting)", height=400
)
below(fig, -0.15)
fig.show()
aucs[aucs["month"] == "pooled"].round(3)

## 3 · Calibration (Fig 3)

Out-of-sample events binned by decile of the predicted probability; the realized win rate of each bin against the
mean prediction. On the diagonal = the probabilities mean what they say. Flat = no information, whatever the AUC
says. (The trade rule only uses the *ranking*, so a miscalibrated but well-ranked model can still select.)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{s}" for s in SIDES], shared_yaxes=True)
for i, side in enumerate(SIDES, start=1):
    top = 0.0
    for m, dash in zip(MODELS, ("solid", "dash"), strict=True):
        o = T[m][side][T[m][side]["p"].notna()]
        cal = rs.calibration(o["p"], o["win"])
        top = max(top, cal["p"].max(), cal["realized"].max())
        fig.add_trace(
            go.Scatter(
                x=cal["p"],
                y=cal["realized"],
                mode="lines+markers",
                name=m,
                line={"color": COLOR[side], "dash": dash},
                showlegend=i == 1,
            ),
            row=1,
            col=i,
        )
    fig.add_trace(
        go.Scatter(
            x=[0, top],
            y=[0, top],
            mode="lines",
            name="perfect",
            showlegend=i == 1,
            line={"color": plots._MUTED, "width": 1},
        ),
        row=1,
        col=i,
    )
plots._base(fig, title="Fig 3 · predicted probability vs realized win rate (deciles)", height=400)
below(fig, -0.15)
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.show()

## 4 · Selection: what the picks earn (Fig 4)

By decile of the prediction: the realized win rate and the mean market-demeaned return, signed as positioned —
gross and after one and two round trips of cost. The line to beat is **"take every event"** (dashed): the model is
only useful where the top deciles earn clearly more than the side's events on average.

In [ ]:
dec = []
for m in MODELS:
    for side in SIDES:
        o = T[m][side][T[m][side]["p"].notna()].copy()
        o["decile"] = pd.qcut(o["p"], 10, labels=False, duplicates="drop")
        for d, g in o.groupby("decile"):
            dec.append(
                {
                    "model": m,
                    "side": side,
                    "decile": int(d) + 1,
                    "events": len(g),
                    "win": g["win"].mean(),
                    "gross bp": 1e4 * g["dret"].mean(),
                    "net 1x bp": 1e4 * (g["dret"].mean() - RT),
                    "net 2x bp": 1e4 * (g["dret"].mean() - 2 * RT),
                }
            )
dec = pd.DataFrame(dec)
fig = make_subplots(
    rows=1, cols=2, subplot_titles=[f"{s}: gross demeaned return by decile" for s in SIDES]
)
for i, side in enumerate(SIDES, start=1):
    allm = 1e4 * T["lr"][side].loc[T["lr"][side]["p"].notna(), "dret"].mean()
    for m, op in zip(MODELS, (0.9, 0.45), strict=True):
        d = dec[(dec["model"] == m) & (dec["side"] == side)]
        fig.add_trace(
            go.Bar(
                x=d["decile"],
                y=d["gross bp"],
                name=m,
                marker_color=COLOR[side],
                opacity=op,
                showlegend=i == 1,
            ),
            row=1,
            col=i,
        )
    fig.add_trace(
        go.Scatter(
            x=[0.5, 10.5],
            y=[allm, allm],
            mode="lines",
            name="take every event",
            showlegend=i == 1,
            line={"color": plots._INK, "dash": "dash", "width": 1},
        ),
        row=1,
        col=i,
    )
    fig.add_trace(
        go.Scatter(
            x=[0.5, 10.5],
            y=[1e4 * RT] * 2,
            mode="lines",
            name="one round trip of cost",
            showlegend=i == 1,
            line={"color": plots._LOSS, "dash": "dot", "width": 1},
        ),
        row=1,
        col=i,
    )
plots._base(
    fig,
    title="Fig 4 · mean demeaned return (bp) by decile of the out-of-sample prediction",
    height=420,
)
below(fig, -0.15)
fig.show()
dec.pivot_table(
    index="decile", columns=["side", "model"], values=["win", "gross bp", "net 2x bp"]
).round(3)

## 5 · What the model learned (Fig 5)

**Left:** the logistic regression's coefficients (standardized features) in every fold, per side — a feature the
model *relies on* keeps its sign month after month; one that flips is noise the model fitted. **Right:** long
against short, averaged over folds: points on the **anti-diagonal** (dashed) mean the two sides learned mirror
images of one *direction* signal; points along the **diagonal** mean both sides learned the SAME thing — which
events will reach *either* barrier, i.e. movement, not direction (a win for one side is a stop for the other, so
only a movement signal can raise both sides' win rates at once). Below: gradient boosting's
permutation importance (drop in AUC when a feature is scrambled) in the last fold.

In [ ]:
from sklearn.inspection import permutation_importance

coef, imp = {}, {}
for side in SIDES:
    rows_ = {}
    t = T["lr"][side]
    X = t[list(S["FEATURES"])].to_numpy(float)
    y = t["win"].to_numpy(float)
    for f in rs.fold_fits(t, ctx, PRM, side, model="lr"):
        rows_[f["month"]] = pd.Series(f["est"][-1].coef_[0], index=S["FEATURES"])
    coef[side] = pd.DataFrame(rows_)
    last_fold = int(T["gbm"][side]["fold"].max())
    last = next(rs.fold_fits(T["gbm"][side], ctx, PRM, side, model="gbm", only=[last_fold]))
    pi = permutation_importance(
        last["est"],
        X[last["test"]],
        y[last["test"]].astype(int),
        scoring="roc_auc",
        n_repeats=5,
        random_state=SEED,
    )
    imp[side] = pd.Series(pi.importances_mean, index=S["FEATURES"])
fig = make_subplots(
    rows=1,
    cols=3,
    column_widths=[0.36, 0.36, 0.28],
    horizontal_spacing=0.09,
    subplot_titles=[
        "long: coefficient per fold",
        "short: coefficient per fold",
        "long vs short (fold mean)",
    ],
)
for i, side in enumerate(SIDES, start=1):
    cf = coef[side]
    fig.add_trace(
        go.Heatmap(
            z=cf.to_numpy(),
            x=cf.columns,
            y=cf.index,
            colorscale=plots.DIVERGING,
            zmid=0,
            zmin=-0.3,
            zmax=0.3,
            showscale=i == 2,
        ),
        row=1,
        col=i,
    )
mean_l, mean_s = coef["long"].mean(axis=1), coef["short"].mean(axis=1)
fig.add_trace(
    go.Scatter(
        x=mean_l,
        y=mean_s,
        mode="markers+text",
        text=mean_l.index,
        textposition="top center",
        textfont={"size": 9},
        showlegend=False,
        marker={"color": plots.SERIES[4], "size": 7},
    ),
    row=1,
    col=3,
)
lim = float(np.nanmax(np.abs(np.r_[mean_l, mean_s]))) * 1.1 or 0.1
fig.add_trace(
    go.Scatter(
        x=[-lim, lim],
        y=[lim, -lim],
        mode="lines",
        showlegend=False,
        line={"color": plots._MUTED, "dash": "dash", "width": 1},
    ),
    row=1,
    col=3,
)
plots._base(
    fig, title="Fig 5 · the logistic model's coefficients (standardized features)", height=560
)
fig.show()
stab = pd.DataFrame(
    {s: (np.sign(coef[s]).eq(np.sign(coef[s].mean(axis=1)), axis=0)).mean(axis=1) for s in SIDES}
)
display(
    pd.DataFrame(
        {
            "long mean": mean_l,
            "short mean": mean_s,
            "long sign kept": stab["long"],
            "short sign kept": stab["short"],
            "long gbm importance": imp["long"],
            "short gbm importance": imp["short"],
        }
    )
    .round(3)
    .sort_values("long gbm importance", ascending=False)
)

## 6 · Controls (the model against what it must beat)

* **take every event** — the side's detector alone, all out-of-sample events.
* **placebo-trained** — the same walk-forward trained on placebo-A bars (same names, months and half hours, no
  burst) and applied to the real events. If it picks as well, the model's skill is generic to these stocks, not
  to bursts.
* **shuffled outcomes** — every event's outcome (label and return) permuted across the whole sample, the model
  re-trained on them and scored **against the same permuted outcomes**: nothing can predict them except a leak
  between training and test months (a failed purge, a duplicated row). Its within-month AUC must sit at 0.5 and its
  picks must earn no more than the average permuted event. If not, every other number is void. (Scored against
  the TRUE labels it would test nothing: a model fit to noise still inherits a random share of the real structure.)
* **label as a feature** — the label appended to the features: the model must find it (AUC ≈ 1). If not, the
  pipeline cannot see a signal even when one exists.

`paired t` compares the model's picks with each control's, day by day (days both traded). On the shuffled row the
two right-hand columns compare ITS picks with all events, both on the permuted outcomes (≈ 0 when nothing leaks).

In [ ]:
pool = rs.placebo_pool(ctx, cooldown_bars=S["_bars"](PRM["cooldown"], panel.bar))


def picked_of(t, wf):
    o = wf["p"].notna().to_numpy()
    pk = wf["picked"].to_numpy() & o
    return t.loc[pk], t.loc[o]


ctrl, ctrl_raw = {}, {}
for side in SIDES:
    t = T["lr"][side]
    real_pk, oos = picked_of(t, t)
    arms = {"model (lr)": real_pk, "take every event": oos}
    pt = rs.placebo_trained(panel, ctx, t, PRM, side, pool, seed=SEED)
    arms["placebo-trained"] = picked_of(t, pt)[0]
    sh, perm = rs.shuffled(t, ctx, PRM, side, seed=SEED)
    so = sh["p"].notna().to_numpy()
    y_perm, r_perm = t["win"].to_numpy(float)[perm], t["dret"].to_numpy(float)[perm]
    sh_pick = sh["picked"].to_numpy() & so
    days = t["session"].to_numpy()
    leak = rs.paired_day_t(r_perm[sh_pick], days[sh_pick], r_perm[so], days[so])
    shuffle_stats = {
        "auc": rs.monthly_auc(sh["p"][so], y_perm[so], sh["fold"][so]),
        "picks − all bp": 1e4 * leak["diff"],
        "t": leak["t"],
        "n": int(so.sum()),
    }
    cn = rs.canary(t, ctx, PRM, side)
    ctrl_raw[side] = {"shuffled": shuffle_stats, "canary": cn, "placebo": pt}
    for name, g in arms.items():
        dt = rs.day_t(g["dret"], g["session"])
        pair = (
            rs.paired_day_t(real_pk["dret"], real_pk["session"], g["dret"], g["session"])
            if name != "model (lr)"
            else {"diff": np.nan, "t": np.nan}
        )
        wf = {"placebo-trained": pt, "model (lr)": t}.get(name)
        o = None if wf is None else wf["p"].notna().to_numpy()
        ctrl[(side, name)] = {
            "events": len(g),
            "win": g["win"].mean(),
            "gross bp": 1e4 * dt["mean"],
            "t": dt["t"],
            "within-month auc": np.nan
            if wf is None
            else rs.monthly_auc(wf["p"][o], t["win"][o], wf["fold"][o]),
            "model − arm bp": 1e4 * pair["diff"],
            "paired t": pair["t"],
        }
    ctrl[(side, "shuffled outcomes (vs permuted)")] = {
        "events": int(sh_pick.sum()),
        "within-month auc": shuffle_stats["auc"],
        "model − arm bp": shuffle_stats["picks − all bp"],
        "paired t": shuffle_stats["t"],
    }
    o = cn["p"].notna().to_numpy()
    ctrl[(side, "label as feature")] = {
        "within-month auc": rs.monthly_auc(cn["p"][o], t["win"][o], cn["fold"][o])
    }
ctrl = pd.DataFrame(ctrl).T
ctrl.round(3)

## 7 · Stability (Fig 7)

The picks' demeaned return after one round trip of cost, month by month; the same without the five names that
contributed most; and the picks' return split into the overnight gaps they held and the rest.

In [ ]:
monthly, stab_rows = [], {}
for side in SIDES:
    pk = T["lr"][side]
    pk = pk[pk["picked"] & pk["p"].notna()]
    for fm, g in pk.groupby("fold"):
        monthly.append(
            {
                "side": side,
                "month": label(fm),
                "picks": len(g),
                "net 1x bp": 1e4 * (g["dret"].mean() - RT),
            }
        )
    top5 = pk.groupby("symbol")["dret"].sum().nlargest(5).index
    rest = pk[~pk["symbol"].isin(top5)]
    parts = rs.split_overnight(panel, ctx, pk[pk["complete"]], side)
    stab_rows[side] = {
        "picks": len(pk),
        "net 1x bp": 1e4 * (pk["dret"].mean() - RT),
        "without top 5 names, net 1x bp": 1e4 * (rest["dret"].mean() - RT),
        "top 5 names": ", ".join(top5),
        "overnight bp": 1e4 * parts["overnight"].mean(),
        "intraday bp": 1e4 * parts["intraday"].mean(),
    }
monthly = pd.DataFrame(monthly)
fig = go.Figure()
for side in SIDES:
    g = monthly[monthly["side"] == side]
    fig.add_trace(go.Bar(x=g["month"], y=g["net 1x bp"], name=side, marker_color=COLOR[side]))
plots._base(
    fig, title="Fig 7 · the picks' demeaned return after one round trip, by month (bp)", height=380
)
below(fig, -0.15)
fig.show()
pd.DataFrame(stab_rows).T

## 8 · Gates a–f and the verdict (per side)

Pre-registered in the README. **e** is checked first: a shuffled-outcome control that "finds" skill means something
leaks, and the run is **VOID** whatever the other gates say. **f** decides which model's picks the verdict reads
(logistic regression unless boosting beats it, paired, t ≥ 2). A side that passes moves to phase 2; a side that
fails is written up in IDEAS.md and dropped from `sides`.

In [ ]:
verdict, checks = {}, []
for side in SIDES:
    sh = ctrl_raw[side]["shuffled"]
    tol = max(0.01, 3 * rs.auc_null_se(T["lr"][side]["win"]))  # no-skill AUC noise, ~ the pooled SE
    sh_auc, sh_t = sh["auc"], sh["t"]
    e_ok = bool(abs(sh_auc - 0.5) <= tol and (not np.isfinite(sh_t) or abs(sh_t) < 2))
    lr_pk = picked_of(T["lr"][side], T["lr"][side])[0]
    gb_pk = picked_of(T["gbm"][side], T["gbm"][side])[0]
    f_pair = rs.paired_day_t(gb_pk["dret"], gb_pk["session"], lr_pk["dret"], lr_pk["session"])
    use = "gbm" if (np.isfinite(f_pair["t"]) and f_pair["t"] >= 2) else "lr"
    pk = gb_pk if use == "gbm" else lr_pk
    oos = T[use][side][T[use][side]["p"].notna()]
    g = rs.day_t(pk["dret"], pk["session"])
    a_ok = bool(g["t"] >= 3 and g["mean"] - 2 * RT > 0)
    vs_all = rs.paired_day_t(pk["dret"], pk["session"], oos["dret"], oos["session"])
    pl_pk = picked_of(T["lr"][side], ctrl_raw[side]["placebo"])[0]
    vs_pl = rs.paired_day_t(pk["dret"], pk["session"], pl_pk["dret"], pl_pk["session"])
    b_ok = bool(vs_all["t"] >= 2 and vs_pl["t"] >= 2)
    by_m = pk.groupby("fold")["dret"].mean() - RT
    c_ok = bool((by_m > 0).sum() >= int(np.ceil(0.7 * len(by_m))) and len(by_m) > 0)
    top5 = pk.groupby("symbol")["dret"].sum().nlargest(5).index
    d_ok = bool(pk.loc[~pk["symbol"].isin(top5), "dret"].mean() - RT > 0)
    gates = {
        "a · t ≥ 3 and net > 0 at 2x cost": a_ok,
        "b · beats every-event and placebo-trained": b_ok,
        "c · ≥ 70% of months positive": c_ok,
        "d · positive without the top 5 names": d_ok,
        "e · shuffled outcomes find nothing": e_ok,
        "f · model used": use,
    }
    passed = a_ok and b_ok and c_ok and d_ok
    verdict[side] = {
        **gates,
        "picks": len(pk),
        "gross bp": 1e4 * g["mean"],
        "t": g["t"],
        "net 2x bp": 1e4 * (g["mean"] - 2 * RT),
        "vs every event t": vs_all["t"],
        "vs placebo-trained t": vs_pl["t"],
        "months positive": f"{int((by_m > 0).sum())}/{len(by_m)}",
        "gbm − lr bp": 1e4 * f_pair["diff"],
        "gbm − lr t": f_pair["t"],
        "shuffled auc (vs permuted)": sh_auc,
        "shuffled tolerance": tol,
        "shuffled picks t": sh_t,
        "VERDICT": "VOID" if not e_ok else ("PASS" if passed else "FAIL"),
    }
    checks.append(
        (f"{side}: verdict", verdict[side]["VERDICT"], "PASS", verdict[side]["VERDICT"] == "PASS")
    )
v = pd.DataFrame(verdict)
display(v)
canary_ok = all(ctrl.loc[(s, "label as feature"), "within-month auc"] > 0.95 for s in SIDES)
purge_ok = bool(audit["unchanged"].all())
checks += [
    ("purge audit: flipped running labels change nothing", purge_ok, True, purge_ok),
    ("label-as-feature canary AUC > 0.95", canary_ok, True, canary_ok),
]
display(rs.checks_table(checks))
rs.scorecard(
    {
        "notebook": "model",
        "study": strat.optuna["study"],
        "names": len(panel.symbols),
        "round_trip_bp": 1e4 * RT,
        "within_month_auc": {
            f"{r['side']}|{r['model']}": r["within-month auc"]
            for _, r in aucs[aucs["month"] == "pooled"].iterrows()
        },
        "verdict": {
            s: {
                k: verdict[s][k]
                for k in (
                    "VERDICT",
                    "f · model used",
                    "picks",
                    "gross bp",
                    "t",
                    "net 2x bp",
                    "vs every event t",
                    "vs placebo-trained t",
                    "months positive",
                    "shuffled auc (vs permuted)",
                )
            }
            for s in SIDES
        },
    }
)